# Code Lab: Model Evaluation & The Bias-Variance Tradeoff

In this execution lab, we step away from building perfectly accurate algorithms, and instead focus on visually demonstrating mathematically *why* models fail. We will dynamically graph Overfitting, perform rigorous Cross-Validation loops, and build an ROC curve from scratch.

In [ ]:
# Step 1: Install require dependencies
!pip install scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_curve, auc

# Ensure consistent reproducibility
np.random.seed(42)

### Visualizing the Bias-Variance Tradeoff (Overfitting)
Decision Trees are notorious for overfitting (High Variance). If we let a tree grow indefinitely, it will memorize every single data point in the training set.

We will graph the Training Error vs the Testing Error as the Tree gets deeper and more complex.

In [ ]:
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

max_depths = range(1, 40)
train_scores = []
test_scores = []

for depth in max_depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    
    # Append the respective accuracies
    train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    test_scores.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(max_depths, train_scores, label="Training Accuracy (Memorization)", color="blue", lw=2)
plt.plot(max_depths, test_scores, label="Testing Accuracy (Real World)", color="red", lw=2)
plt.axvline(x=6, color='gray', linestyle='--', label='Perfect Complexity Sweet Spot')

plt.title("The Bias-Variance Tradeoff Graph")
plt.xlabel("Model Complexity (Tree Depth)")
plt.ylabel("Accuracy Score")
plt.legend()
plt.grid()
plt.show()

**Analysis:** Look at the graph. On the left side (Depth = 1), the model is Underfitting (High Bias) and scores terribly. As depth increases, both lines go up until **Depth 6**. Beyond Depth 6, the Blue line continues rising towards 100%, but the Red line drops! The model has started to **Overfit (High Variance)**. It is memorizing noise.

### Rigorous Cross-Validation (K-Fold)
A single train-test split (like above) can be mathematically lucky or unlucky. We use K-Fold Validation to prove the exact accuracy without a doubt.

In [ ]:
# Re-instantiate the model at the perfect depth we found above
sweet_spot_model = DecisionTreeClassifier(max_depth=6, random_state=42)

# Perform 5-Fold Stratified Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(sweet_spot_model, X, y, cv=skf, scoring='accuracy')

print("Resulting scores from the 5 independent testing quizzes:")
print(cv_scores)
print(f"\nFINAL TRUTH: Mathematical Accuracy is exactly: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

### Plotting the ROC-AUC Curve
The physical plotting of True Positive Rate vs False Positive Rate. A model guessing randomly produces a 45-degree diagonal line (AUC 0.50).

In [ ]:
sweet_spot_model.fit(X_train, y_train)

# Notice we use 'predict_proba', not 'predict'. ROC relies on raw mathematical probabilities!
probabilities = sweet_spot_model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, probabilities)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing (AUC 0.50)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (Accidental False Alarms)')
plt.ylabel('True Positive Rate (Correct Detections)')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid()
plt.show()